In [6]:
import os
import ROOT
import pandas as pd
import numpy as np

# =====================================================
# SETTINGS
# =====================================================
working_directory = "/root/geant4/detector/Lung_ICRP"
os.chdir(working_directory)

file_path = "Lung_TissueTumor_AK01_460_deg.root"
tree_name = "t"

# Incident gamma energy, keV
E0 = 662.0

# Electron rest energy, keV
ME = 511.0

# Incident beam direction
# /gps/direction 0 -1 0
incident_dir = np.array([0.0, -1.0, 0.0])

# Volumes where scattering of interest occurs
scatter_volumes = [5, 3]

# Detector volume
detector_volume = 460

# Gamma PDG
gamma_pdg = 22

# Your Compton process code
compton_process = 2013

# Output
excel_file = "Lung_TissueTumor_AK01_460.xlsx"

print("Working directory:", os.getcwd())
print("ROOT file:", os.path.abspath(file_path))


# =====================================================
# OPEN ROOT FILE
# =====================================================
root_file = ROOT.TFile.Open(file_path)

if not root_file or root_file.IsZombie():
    raise OSError(
        f"Could not open ROOT file: {os.path.abspath(file_path)}"
    )

tree = root_file.Get(tree_name)

if not tree:
    root_file.Close()
    raise KeyError(
        f"TTree '{tree_name}' not found"
    )

print("Total TTree entries:", tree.GetEntries())


# =====================================================
# SAFE VECTOR VALUE
# =====================================================
def safe_value(vector, index, default=np.nan):
    if index < len(vector):
        return vector[index]
    return default


# =====================================================
# ANGLE FROM MOMENTUM DIRECTION
# =====================================================
def theta_from_momentum(px, py, pz):

    pvec = np.array(
        [px, py, pz],
        dtype=float
    )

    pmag = np.linalg.norm(pvec)

    if pmag == 0:
        return np.nan

    cos_theta = np.dot(
        pvec,
        incident_dir
    ) / pmag

    cos_theta = np.clip(
        cos_theta,
        -1.0,
        1.0
    )

    return np.degrees(
        np.arccos(cos_theta)
    )


# =====================================================
# ANGLE FROM COMPTON ENERGY
# =====================================================
def theta_from_k(k):

    if not np.isfinite(k):
        return np.nan

    if k <= 0:
        return np.nan

    cos_theta = (
        1.0
        - ME
        * (
            (1.0 / k)
            - (1.0 / E0)
        )
    )

    # If outside physical single-Compton range
    if cos_theta < -1.0 or cos_theta > 1.0:
        return np.nan

    return np.degrees(
        np.arccos(cos_theta)
    )


# =====================================================
# STORAGE
# =====================================================
matched_rows = []


# =====================================================
# LOOP THROUGH EVENTS
# =====================================================
for tree_entry, event in enumerate(tree):

    n = len(event.pdg)

    # -------------------------------------------------
    # Collect all records for this TTree entry
    # -------------------------------------------------
    records = []

    for i in range(n):

        if i >= len(event.vlm):
            continue

        if i >= len(event.trk):
            continue

        pdg = int(event.pdg[i])
        vlm = int(event.vlm[i])
        trk = int(event.trk[i])

        record = {
            "index": i,
            "pdg": pdg,
            "vlm": vlm,
            "trk": trk,

            "pro": int(
                safe_value(
                    event.pro,
                    i,
                    -1
                )
            ),

            "stp": int(
                safe_value(
                    event.stp,
                    i,
                    -1
                )
            ),

            "k": float(
                safe_value(
                    event.k,
                    i
                )
            ),

            "et": float(
                safe_value(
                    event.et,
                    i
                )
            ),

            "de": float(
                safe_value(
                    event.de,
                    i
                )
            ),

            "x": float(
                safe_value(
                    event.x,
                    i
                )
            ),

            "y": float(
                safe_value(
                    event.y,
                    i
                )
            ),

            "z": float(
                safe_value(
                    event.z,
                    i
                )
            ),

            "px": float(
                safe_value(
                    event.px,
                    i
                )
            ),

            "py": float(
                safe_value(
                    event.py,
                    i
                )
            ),

            "pz": float(
                safe_value(
                    event.pz,
                    i
                )
            )
        }

        records.append(record)

    # =================================================
    # FIND COMPTON SCATTER RECORDS IN VOLUME 11 OR 3
    # =================================================
    scatter_records = []

    for rec in records:

        if rec["pdg"] != gamma_pdg:
            continue

        if rec["vlm"] not in scatter_volumes:
            continue

        if rec["pro"] != compton_process:
            continue

        scatter_records.append(rec)

    # =================================================
    # FOR EACH SCATTER, FIND SAME TRACK LATER IN VLM 460
    # =================================================
    for scatter in scatter_records:

        scatter_index = scatter["index"]
        scatter_track = scatter["trk"]

        detector_candidates = []

        for rec in records:

            # Must be later in the stored sequence
            if rec["index"] <= scatter_index:
                continue

            # Same gamma track
            if rec["trk"] != scatter_track:
                continue

            if rec["pdg"] != gamma_pdg:
                continue

            # Must reach detector 445
            if rec["vlm"] != detector_volume:
                continue

            detector_candidates.append(rec)

        # No detector hit from this scattered gamma
        if len(detector_candidates) == 0:
            continue

        # -------------------------------------------------
        # Use first occurrence in detector
        # -------------------------------------------------
        det = detector_candidates[0]

        # =================================================
        # CALCULATE ANGLES AT SCATTER
        # =================================================
        theta_p_scatter = theta_from_momentum(
            scatter["px"],
            scatter["py"],
            scatter["pz"]
        )

        theta_k_scatter = theta_from_k(
            scatter["k"]
        )

        # =================================================
        # CALCULATE ANGLE FROM MOMENTUM AT DETECTOR
        # =================================================
        theta_p_detector = theta_from_momentum(
            det["px"],
            det["py"],
            det["pz"]
        )

        theta_k_detector = theta_from_k(
            det["k"]
        )

        # =================================================
        # ENERGY LOSS
        # =================================================
        energy_loss_scatter = (
            E0 - scatter["k"]
            if np.isfinite(scatter["k"])
            else np.nan
        )

        # =================================================
        # SAVE MATCH
        # =================================================
        matched_rows.append({

            # Event identity
            "tree_entry": tree_entry,
            "trk": scatter_track,

            # ---------------------------------------------
            # SCATTER RECORD
            # ---------------------------------------------
            "scatter_volume": scatter["vlm"],
            "scatter_vector_index": scatter["index"],
            "scatter_pro": scatter["pro"],
            "scatter_stp": scatter["stp"],

            "scatter_k_keV": scatter["k"],
            "scatter_energy_loss_keV":
                energy_loss_scatter,

            "scatter_et": scatter["et"],
            "scatter_de": scatter["de"],

            "scatter_x": scatter["x"],
            "scatter_y": scatter["y"],
            "scatter_z": scatter["z"],

            "scatter_px": scatter["px"],
            "scatter_py": scatter["py"],
            "scatter_pz": scatter["pz"],

            "theta_p_scatter_deg":
                theta_p_scatter,

            "theta_k_scatter_deg":
                theta_k_scatter,

            # ---------------------------------------------
            # DETECTOR RECORD
            # ---------------------------------------------
            "detector_volume": detector_volume,
            "detector_vector_index": det["index"],
            "detector_pro": det["pro"],
            "detector_stp": det["stp"],

            "detector_k_keV": det["k"],
            "detector_et": det["et"],
            "detector_de": det["de"],

            "detector_x": det["x"],
            "detector_y": det["y"],
            "detector_z": det["z"],

            "detector_px": det["px"],
            "detector_py": det["py"],
            "detector_pz": det["pz"],

            "theta_p_detector_deg":
                theta_p_detector,

            "theta_k_detector_deg":
                theta_k_detector,

            # ---------------------------------------------
            # ANGLE COMPARISON
            # ---------------------------------------------
            "theta_difference_scatter_deg":
                (
                    theta_p_scatter
                    - theta_k_scatter
                    if np.isfinite(theta_p_scatter)
                    and np.isfinite(theta_k_scatter)
                    else np.nan
                ),

            "theta_difference_detector_deg":
                (
                    theta_p_detector
                    - theta_k_detector
                    if np.isfinite(theta_p_detector)
                    and np.isfinite(theta_k_detector)
                    else np.nan
                )
        })


# =====================================================
# CREATE DATAFRAME
# =====================================================
df_matches = pd.DataFrame(matched_rows)

print("\n========================================")
print("MATCHED GAMMAS")
print("========================================")

print(
    "Total gammas scattered in volume 5 or 3 "
    "and later reaching volume 460:",
    len(df_matches)
)


# =====================================================
# SEPARATE VOLUME 5 AND 3
# =====================================================
if len(df_matches) > 0:

    df_vlm5 = df_matches[
        df_matches["scatter_volume"] == 5
    ].copy()

    df_vlm3 = df_matches[
        df_matches["scatter_volume"] == 3
    ].copy()

else:

    df_vlm5 = pd.DataFrame()
    df_vlm3 = pd.DataFrame()


print("From volume 5:", len(df_vlm5))
print("From volume 3:", len(df_vlm3))


# =====================================================
# OPTIONAL: DETECTOR ANGULAR ACCEPTANCE
# 30 +/- 11 degrees
# =====================================================
theta_min = 53.09
theta_max = 66.91

if len(df_matches) > 0:

    df_detector_angle = df_matches[
        (df_matches["theta_p_detector_deg"] >= theta_min)
        &
        (df_matches["theta_p_detector_deg"] <= theta_max)
    ].copy()

else:

    df_detector_angle = pd.DataFrame()


print(
    "Matched photons with detector direction "
    "between 53.09 and 66.91 deg:",
    len(df_detector_angle)
)


# =====================================================
# OPTIONAL: CLEAN SINGLE-SCATTER-LIKE SAMPLE
#
# Require theta from momentum and theta from k
# to approximately agree.
# =====================================================
angle_tolerance = 1.0   # degrees

if len(df_matches) > 0:

    df_single_like = df_matches[
        np.isfinite(
            df_matches["theta_k_scatter_deg"]
        )
        &
        (
            np.abs(
                df_matches["theta_p_scatter_deg"]
                -
                df_matches["theta_k_scatter_deg"]
            )
            <= angle_tolerance
        )
    ].copy()

else:

    df_single_like = pd.DataFrame()


print(
    "Single-Compton-like matches "
    f"(|theta_p-theta_k| <= {angle_tolerance} deg):",
    len(df_single_like)
)


# =====================================================
# SAVE EXCEL
# =====================================================
with pd.ExcelWriter(
    excel_file,
    engine="openpyxl"
) as writer:

    df_matches.to_excel(
        writer,
        sheet_name="All_matches",
        index=False
    )

    df_vlm5.to_excel(
        writer,
        sheet_name="Scatter_Volume_5",
        index=False
    )

    df_vlm3.to_excel(
        writer,
        sheet_name="Scatter_Volume_3",
        index=False
    )

    df_detector_angle.to_excel(
        writer,
        sheet_name="Detector_53_66_deg",
        index=False
    )

    df_single_like.to_excel(
        writer,
        sheet_name="Single_Compton_like",
        index=False
    )


# =====================================================
# DISPLAY
# =====================================================
print("\nExcel file saved:")
print(os.path.abspath(excel_file))

if len(df_matches) > 0:

    display(
        df_matches[
            [
                "tree_entry",
                "trk",
                "scatter_volume",
                "scatter_k_keV",
                "theta_p_scatter_deg",
                "theta_k_scatter_deg",
                "detector_k_keV",
                "theta_p_detector_deg",
                "theta_k_detector_deg"
            ]
        ].head(30)
    )

else:

    print(
        "\nNo matching gamma tracks were found."
    )


# =====================================================
# CLOSE ROOT FILE
# =====================================================
root_file.Close()

Working directory: /root/geant4/detector/Lung_ICRP
ROOT file: /root/geant4/detector/Lung_ICRP/Lung_TissueTumor_AK01_460_deg.root
Total TTree entries: 1000000

MATCHED GAMMAS
Total gammas scattered in volume 5 or 3 and later reaching volume 460: 201
From volume 5: 160
From volume 3: 41
Matched photons with detector direction between 53.09 and 66.91 deg: 129
Single-Compton-like matches (|theta_p-theta_k| <= 1.0 deg): 182

Excel file saved:
/root/geant4/detector/Lung_ICRP/Lung_TissueTumor_AK01_460.xlsx


,tree_entry,trk,scatter_volume,scatter_k_keV,theta_p_scatter_deg,theta_k_scatter_deg,detector_k_keV,theta_p_detector_deg,theta_k_detector_deg
0,2625,1,5,367.155139,67.658735,67.658817,367.155139,67.658735,67.658817
1,8388,1,3,399.530012,60.468525,60.468596,399.530012,60.468525,60.468596
2,10350,1,5,428.945158,54.506560,54.506623,428.945158,54.506560,54.506623
3,35345,1,5,405.579676,59.204240,59.204309,405.579676,59.204240,59.204309
4,35514,1,3,390.932952,62.304342,62.304416,390.932952,62.304342,62.304416
5,46701,1,5,390.234346,62.455661,62.455735,390.234346,62.455661,62.455735
6,50322,1,5,390.865336,62.318973,62.319047,202.095810,72.228896,139.165402
7,56096,1,5,416.949685,56.883940,56.884006,416.949685,56.883940,56.884006
8,59723,1,5,374.472931,65.963409,65.963489,374.472931,65.963409,65.963489
9,72122,1,5,212.494174,129.261816,129.262073,0.000000,67.895598,NaN
